In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE

print('All libraries imported successfully')

In [ ]:
#Update this path to wherever your CSV is saved
df = pd.read_csv('last_clean.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Check target class distribution
print('Addiction Level Distribution:')
print(df['Addiction_Level'].value_counts())

plt.figure(figsize=(10, 5))
df['Addiction_Level'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Addiction Level Distribution (Before SMOTE)')
plt.xlabel('Addiction Level')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Encode Gender: M=1, F=0
df['Gender_Num'] = df['Gender'].map({'M': 1, 'F': 0})

# Feature columns used for training
FEATURE_COLS = [
    'Gender_Num',
    'AScore', 'Nscore', 'Oscore', 'Escore', 'Cscore',
    'Impulsive', 'SS',
    'Nicotine_Num', 'Meth_Num', 'Heroin_Num',
    'Cannabis_Num', 'Coke_Num', 'Alcohol_Num',
    'High_Impulsive', 'Low_Control', 'Max_Drug', 'Risky_Personality'
]

X = df[FEATURE_COLS].copy()
y = df['Addiction_Level']

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print('Classes:', label_encoder.classes_)
print('Feature matrix shape:', X.shape)

In [ ]:
# Fix class imbalance using SMOTE
sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X, y_encoded)

print('Class counts after SMOTE:')
unique, counts = np.unique(y_resampled, return_counts=True)
for u, c in zip(label_encoder.classes_, counts):
    print(f'  {u}: {c}')

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')

In [ ]:
# Train model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print(' Model trained successfully!')

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'🎯 Test Accuracy: {accuracy * 100:.2f}%\n')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importances = model.feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue')
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
#  DRUG FREQUENCY MAPPING  (Q0–Q5, multiple choice)

DRUG_FREQ_MAP = {
    'Never':      0,
    'Rarely':     1,
    'Sometimes':  2,
    'Often':      4,
    'Very Often': 6
}

TRAIT_KEYWORDS = {
    # Nscore: HIGH = anxious, unstable, worried
    'Nscore': {
        'positive': ['anxious', 'stressed', 'worry', 'worried', 'nervous',
                     'unstable', 'overwhelmed', 'panic', 'overthink', 'overthinking',
                     'mood swings', 'emotional', 'upset', 'yes', 'often', 'frequently',
                     'always', 'a lot', 'very', 'deeply', 'racing thoughts'],
        'negative': ['calm', 'relaxed', 'stable', 'no', 'never', 'rarely',
                     'not really', 'not much', 'barely', 'seldom', 'fine', 'okay']
    },
    # Escore: HIGH = social, energized by others
    'Escore': {
        'positive': ['enjoy', 'love', 'energized', 'social', 'people', 'gatherings',
                     'outgoing', 'active', 'fun', 'motivated', 'yes', 'often',
                     'always', 'definitely', 'very much', 'group', 'friends'],
        'negative': ['no', 'never', 'rarely', 'prefer alone', 'introvert',
                     'quiet', 'shy', 'uncomfortable', 'dislike', 'tired',
                     'drain', 'avoid', 'not really']
    },
    # Oscore: HIGH = curious, open, creative, risk-taker
    'Oscore': {
        'positive': ['new', 'experience', 'curious', 'creative', 'explore',
                     'enjoy', 'love', 'adventure', 'different', 'learn',
                     'open', 'yes', 'often', 'try', 'risk', 'challenge'],
        'negative': ['no', 'never', 'prefer routine', 'boring', 'same',
                     'not interested', 'avoid', 'rarely', 'familiar', 'safe']
    },
    # AScore: HIGH = cooperative, empathetic, conflict-avoidant
    'AScore': {
        'positive': ['cooperative', 'empathetic', 'understand', 'help', 'kind',
                     'peaceful', 'avoid conflict', 'support', 'care', 'sensitive',
                     'yes', 'often', 'always', 'definitely', 'friendly'],
        'negative': ['no', 'never', 'argue', 'conflict', 'selfish', 'aggressive',
                     'don\'t care', 'ignore', 'rarely', 'not really']
    },
    # Cscore: HIGH = organized, disciplined, responsible
    'Cscore': {
        'positive': ['plan', 'organized', 'routine', 'disciplined', 'responsible',
                     'schedule', 'focused', 'on time', 'yes', 'always', 'often',
                     'definitely', 'prepare', 'goal'],
        'negative': ['no', 'never', 'disorganized', 'lazy', 'procrastinate',
                     'forget', 'messy', 'rarely', 'not really', 'skip', 'miss']
    },
    # Impulsive: HIGH = acts without thinking
    'Impulsive': {
        'positive': ['impulsive', 'without thinking', 'react', 'regret', 'quick',
                     'immediately', 'suddenly', 'burst', 'yes', 'often', 'always',
                     'can\'t control', 'instantly', 'emotion', 'snap'],
        'negative': ['no', 'never', 'think first', 'careful', 'consider',
                     'plan', 'rarely', 'calm', 'patient', 'wait', 'not really']
    },
    # SS: HIGH = thrill-seeker, loves excitement & danger
    'SS': {
        'positive': ['thrill', 'exciting', 'risky', 'danger', 'adventure',
                     'extreme', 'adrenaline', 'love', 'enjoy', 'fun', 'yes',
                     'often', 'always', 'definitely', 'dare', 'bold'],
        'negative': ['no', 'never', 'safe', 'boring', 'avoid', 'not really',
                     'rarely', 'prefer safe', 'scared', 'cautious', 'fear']
    }
}


def text_to_personality_score(answer_text: str, trait: str) -> float:
    """
    Convert a free-text answer to a normalized personality score.
    Uses keyword matching to determine trait strength.
    Returns a float in range [-2.0, +2.0] (z-score style).
    """
    if not answer_text or not answer_text.strip():
        return 0.0
    
    text = answer_text.lower().strip()
    keywords = TRAIT_KEYWORDS.get(trait, {'positive': [], 'negative': []})
    
    pos_hits = sum(1 for kw in keywords['positive'] if kw in text)
    neg_hits = sum(1 for kw in keywords['negative'] if kw in text)
    
    net = pos_hits - neg_hits
    
    # Scale to roughly [-2, 2]
    if net > 0:
        return min(2.0, 0.5 + net * 0.4)
    elif net < 0:
        return max(-2.0, -0.5 + net * 0.4)
    else:
        return 0.0


def average_trait_score(ans_a: str, ans_b: str, trait: str) -> float:
    """Average score from two questions that measure the same trait."""
    s1 = text_to_personality_score(ans_a, trait)
    s2 = text_to_personality_score(ans_b, trait)
    return round((s1 + s2) / 2.0, 5)


print('Personality score converter ready')

# Quick test
print('\n--- Quick test ---')
print('Nscore (anxious):', text_to_personality_score('I overthink at night and feel very nervous', 'Nscore'))
print('Nscore (calm):   ', text_to_personality_score('No I am calm and relaxed', 'Nscore'))
print('Escore (social): ', text_to_personality_score('I love being with friends and feel energized', 'Escore'))
print('Impulsive (high):', text_to_personality_score('Yes I often react without thinking and regret later', 'Impulsive'))
print('SS (thrill):     ', text_to_personality_score('I love exciting and risky adventures', 'SS'))

In [ ]:
def predict_addiction_from_questionnaire(
    answers: dict,
    gender: str = 'M'
) -> dict:
    """
    Main prediction function.
    
    Parameters:
    -----------
    answers : dict  — keys are question indices (0–19), values are answer strings
                      Q0–Q5:  'Never' / 'Rarely' / 'Sometimes' / 'Often' / 'Very Often'
                      Q6–Q19: free text (from VoiceTextInput)
    gender  : str   — 'M' or 'F'
    
    Returns:
    --------
    dict with:
        - addiction_level   : predicted label
        - confidence        : probability of predicted class
        - all_probabilities : dict of all class probabilities
        - personality_scores: derived scores shown to user
        - drug_scores       : drug frequency scores used
    """
    
    # 1. DRUG FREQUENCY (Q0–Q5)
    alcohol_raw  = answers.get(0, 'Never')
    cannabis_raw = answers.get(1, 'Never')
    coke_raw     = answers.get(2, 'Never')
    heroin_raw   = answers.get(3, 'Never')
    meth_raw     = answers.get(4, 'Never')
    nicotine_raw = answers.get(5, 'Never')

    alcohol_num  = DRUG_FREQ_MAP.get(alcohol_raw, 0)
    cannabis_num = DRUG_FREQ_MAP.get(cannabis_raw, 0)
    coke_num     = DRUG_FREQ_MAP.get(coke_raw, 0)
    heroin_num   = DRUG_FREQ_MAP.get(heroin_raw, 0)
    meth_num     = DRUG_FREQ_MAP.get(meth_raw, 0)
    nicotine_num = DRUG_FREQ_MAP.get(nicotine_raw, 0)

    # 2. PERSONALITY SCORES FROM TEXT (Q6–Q19)
    nscore    = average_trait_score(answers.get(6,''), answers.get(7,''), 'Nscore')
    escore    = average_trait_score(answers.get(8,''), answers.get(9,''), 'Escore')
    oscore    = average_trait_score(answers.get(10,''), answers.get(11,''), 'Oscore')
    ascore    = average_trait_score(answers.get(12,''), answers.get(13,''), 'AScore')
    cscore    = average_trait_score(answers.get(14,''), answers.get(15,''), 'Cscore')
    impulsive = average_trait_score(answers.get(16,''), answers.get(17,''), 'Impulsive')
    ss        = average_trait_score(answers.get(18,''), answers.get(19,''), 'SS')

    # 3. DERIVED / ENGINEERED FEATURES
    gender_num        = 1 if gender.upper() == 'M' else 0
    high_impulsive    = 1 if impulsive > 0.5 else 0
    low_control       = 1 if cscore < -0.5 else 0
    max_drug          = max(alcohol_num, cannabis_num, coke_num, heroin_num, meth_num, nicotine_num)
    risky_personality = 1 if (oscore > 0.5 and ss > 0.5) else 0

    #  4. BUILD FEATURE ROW
    features = pd.DataFrame([{
        'Gender_Num':        gender_num,
        'AScore':            ascore,
        'Nscore':            nscore,
        'Oscore':            oscore,
        'Escore':            escore,
        'Cscore':            cscore,
        'Impulsive':         impulsive,
        'SS':                ss,
        'Nicotine_Num':      nicotine_num,
        'Meth_Num':          meth_num,
        'Heroin_Num':        heroin_num,
        'Cannabis_Num':      cannabis_num,
        'Coke_Num':          coke_num,
        'Alcohol_Num':       alcohol_num,
        'High_Impulsive':    high_impulsive,
        'Low_Control':       low_control,
        'Max_Drug':          max_drug,
        'Risky_Personality': risky_personality
    }])
    features = features[FEATURE_COLS]   # ensure column order matches training

    # 5. PREDICT
    pred_encoded  = model.predict(features)[0]
    pred_label    = label_encoder.inverse_transform([pred_encoded])[0]
    probabilities = model.predict_proba(features)[0]
    confidence    = round(float(probabilities[pred_encoded]) * 100, 2)

    all_probs = {
        label_encoder.classes_[i]: round(float(p) * 100, 2)
        for i, p in enumerate(probabilities)
    }

    return {
        'addiction_level':    pred_label,
        'confidence':         f'{confidence}%',
        'all_probabilities':  all_probs,
        'personality_scores': {
            'Nscore (Neuroticism)':      nscore,
            'Escore (Extraversion)':     escore,
            'Oscore (Openness)':         oscore,
            'AScore (Agreeableness)':    ascore,
            'Cscore (Conscientiousness)': cscore,
            'Impulsive':                 impulsive,
            'SS (Sensation Seeking)':    ss
        },
        'drug_scores': {
            'Alcohol':  alcohol_num,
            'Cannabis': cannabis_num,
            'Cocaine':  coke_num,
            'Heroin':   heroin_num,
            'Meth':     meth_num,
            'Nicotine': nicotine_num
        }
    }


print('predict_addiction_from_questionnaire() ready')

In [ ]:
# TEST CASE 1: High Risk User
answers_high_risk = {
    # Drug frequency (Q0–Q5)
    0: 'Very Often',   # Alcohol
    1: 'Often',        # Cannabis
    2: 'Sometimes',    # Cocaine
    3: 'Rarely',       # Heroin
    4: 'Never',        # Meth
    5: 'Very Often',   # Nicotine

    # Personality text answers (Q6–Q19)
    6:  'I feel very anxious and stressed all the time, I overthink everything',
    7:  'Yes I am emotionally unstable and my mood swings are very severe',
    8:  'I enjoy being with people and go to gatherings often',
    9:  'Yes talking with friends always makes me feel energized and active',
    10: 'I love trying new and unusual experiences especially risky ones',
    11: 'Yes I am very open to taking risks and exploring new ideas',
    12: 'I try to be cooperative and empathetic but sometimes I struggle',
    13: 'I often avoid conflicts and prefer peaceful resolution',
    14: 'I rarely plan things I am not very organized',
    15: 'No I am not very disciplined and I miss deadlines often',
    16: 'Yes I often act without thinking and regret later instantly',
    17: 'I make very quick decisions without any planning at all',
    18: 'I love thrilling and exciting risky activities they are fun',
    19: 'Yes I seek excitement even if it involves danger and risk'
}

result1 = predict_addiction_from_questionnaire(answers_high_risk, gender='M')

print('=' * 55)
print('TEST CASE 1: High Risk User')
print('=' * 55)
print(f"\n🔴 Predicted Level : {result1['addiction_level']}")
print(f"📊 Confidence      : {result1['confidence']}")
print(f"\n📋 Personality Scores Derived:")
for k, v in result1['personality_scores'].items():
    print(f"   {k:<35}: {v:>6.3f}")
print(f"\n💊 Drug Scores:")
for k, v in result1['drug_scores'].items():
    print(f"   {k:<12}: {v}")
print(f"\n📈 All Probabilities:")
for level, prob in sorted(result1['all_probabilities'].items(), key=lambda x: -x[1]):
    bar = '█' * int(prob / 5)
    print(f"   {level:<30}: {prob:>6.2f}% {bar}")

In [ ]:
#Save model

joblib.dump(model,         'addiction_rf_model.pkl')
joblib.dump(label_encoder, 'addiction_label_encoder.pkl')
joblib.dump(FEATURE_COLS,  'feature_columns.pkl')

print('Saved:')
print('   addiction_rf_model.pkl')
print('   addiction_label_encoder.pkl')
print('   feature_columns.pkl')

In [ ]:
#Step 10: Flask API Ready for Your Backend
flask_code = '''
from flask import Flask, request, jsonify
import joblib
import pandas as pd

# ── Load saved model components ──────────────────────────────
model         = joblib.load('addiction_rf_model.pkl')
label_encoder = joblib.load('addiction_label_encoder.pkl')
FEATURE_COLS  = joblib.load('feature_columns.pkl')

# ── Copy the DRUG_FREQ_MAP, TRAIT_KEYWORDS, text_to_personality_score,
#    average_trait_score, and predict_addiction_from_questionnaire
#    functions here from the notebook ──────────────────────────

app = Flask(__name__)

@app.route("/api/questionnaire/submit", methods=["POST"])
def submit_questionnaire():
    data    = request.get_json()
    userId  = data.get("userId")
    answers = data.get("answers", {})
    gender  = data.get("gender", "M")

    # Convert string keys to int (JSON keys are always strings)
    answers = {int(k): v for k, v in answers.items()}

    result = predict_addiction_from_questionnaire(answers, gender=gender)

    return jsonify({
        "userId":          userId,
        "addiction_level": result["addiction_level"],
        "confidence":      result["confidence"],
        "personality":     result["personality_scores"],
        "drug_scores":     result["drug_scores"]
    })

if __name__ == "__main__":
    app.run(debug=True, port=5000)
'''

with open('app.py', 'w') as f:
    f.write(flask_code)

print('app.py written — run with: python app.py')
print('   Endpoint: POST /api/questionnaire/submit')
print('   Body: { userId, answers: {0..19: string}, gender: M/F }')